<a href="https://colab.research.google.com/github/GanyaGit/GreenInfer/blob/main/greeninfer_week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install requests pandas plotly

In [2]:
import requests
import pandas as pd
from datetime import datetime


# GreenInfer — Week 1
# Electricity Maps API client


apiKey = "em_HZVHj4DhVXJYp8RDfRFxnrpbqXEQNFGW"
Headers = {"auth-token": apiKey}
baseUrl = "https://api.electricitymap.org/v3"

def get_carbon_intensity(zone: str) -> dict:
    """
    Get real-time carbon intensity (gCO2eq/kWh) for a region.
    zone examples: "CN" (China), "US-CAL-CISO" (California), "DE" (Germany)
    """
    url = f"{baseUrl}/carbon-intensity/latest?zone={zone}"
    response = requests.get(url, headers=Headers)

    if response.status_code == 200:
        data = response.json()
        return {
            "zone": zone,
            "carbon_intensity_gco2_per_kwh": data["carbonIntensity"],
            "timestamp": data["datetime"],
            "fetched_at": datetime.utcnow().isoformat()
        }
    else:
        return {"error": response.status_code, "zone": zone}


def get_power_breakdown(zone: str) -> dict:
    """
    Get how electricity is being generated right now — solar, coal, wind etc.
    """
    url = f"{baseUrl}/power-breakdown/latest?zone={zone}"
    response = requests.get(url, headers=Headers)

    if response.status_code == 200:
        data = response.json()
        return {
            "zone": zone,
            "renewable_percentage": data.get("renewablePercentage"),
            "fossil_fuel_percentage": data.get("fossilFuelPercentage"),
            "power_sources": data.get("powerProductionBreakdown", {})
        }
    else:
        return {"error": response.status_code, "zone": zone}


# Test it with 3 major data center regions
zones = ["CN", "US-CAL-CISO", "DE"]

results = []
for zone in zones:
    ci = get_carbon_intensity(zone)
    pb = get_power_breakdown(zone)
    results.append({**ci, **pb})
    print(f"✅ {zone} — {ci.get('carbon_intensity_gco2_per_kwh')} gCO2/kWh")

# Save to DataFrame
df = pd.DataFrame(results)
print("\nThe first dataset")
print(df[["zone", "carbon_intensity_gco2_per_kwh","renewable_percentage", "fossil_fuel_percentage"]])

# Save to CSV — this goes into HDFS later
df.to_csv("electricity_data_day1.csv", index=False)
print("\n✅ Saved to electricity_data_day1.csv")

/tmp/ipykernel_1759/1804399870.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fetched_at": datetime.utcnow().isoformat()


✅ CN — 520 gCO2/kWh
✅ US-CAL-CISO — 105 gCO2/kWh
✅ DE — 363 gCO2/kWh

The first dataset
          zone  carbon_intensity_gco2_per_kwh  renewable_percentage  \
0           CN                            520                    31   
1  US-CAL-CISO                            105                    77   
2           DE                            363                    59   

  fossil_fuel_percentage  
0                   None  
1                   None  
2                   None  

✅ Saved to electricity_data_day1.csv


In [3]:
# ================================================
# GreenInfer — Week 1 | Cell 3
# China Data Portal — Official NBS Energy Data
# No API key needed!
# ================================================

import requests
import pandas as pd
from datetime import datetime

BASE = "https://chinadata.live/api/v2"

print("🔄 Fetching China energy data from NBS...\n")

# ── 1. Monthly Electricity Generation ──────────────
try:
    r1 = requests.get(f"{BASE}/data/china-electricity-generation", timeout=10)
    df_elec = pd.DataFrame(r1.json()["data"])
    df_elec.to_csv("china_electricity_generation.csv", index=False)
    print(f"✅ Electricity Generation — {len(df_elec)} records")
    print(df_elec.tail(3).to_string(index=False))
except Exception as e:
    print(f"❌ Electricity Generation failed: {e}")

print()

# ── 2. Energy Consumption by Source ────────────────
try:
    r2 = requests.get(f"{BASE}/data/china-energy-consumption-by-source", timeout=10)
    df_mix = pd.DataFrame(r2.json()["data"])
    df_mix.to_csv("china_energy_mix.csv", index=False)
    print(f"✅ Energy Source Mix — {len(df_mix)} records")
    print(df_mix.tail(3).to_string(index=False))
except Exception as e:
    print(f"❌ Energy Mix failed: {e}")

print()

# ── 3. Solar Installed Capacity ─────────────────────
try:
    r3 = requests.get(f"{BASE}/data/china-power-installed-capacity-solar", timeout=10)
    df_solar = pd.DataFrame(r3.json()["data"])
    df_solar.to_csv("china_solar_capacity.csv", index=False)
    print(f"✅ Solar Capacity — {len(df_solar)} records")
    print(df_solar.tail(3).to_string(index=False))
except Exception as e:
    print(f"❌ Solar Capacity failed: {e}")

print()

# ── 4. Wind Installed Capacity ──────────────────────
try:
    r4 = requests.get(f"{BASE}/data/china-power-installed-capacity-wind", timeout=10)
    df_wind = pd.DataFrame(r4.json()["data"])
    df_wind.to_csv("china_wind_capacity.csv", index=False)
    print(f"✅ Wind Capacity — {len(df_wind)} records")
    print(df_wind.tail(3).to_string(index=False))
except Exception as e:
    print(f"❌ Wind Capacity failed: {e}")

print()

# ── Summary ─────────────────────────────────────────
print("=" * 50)
print("📊 WEEK 1 DATASET SUMMARY")
print("=" * 50)
print(f"{'Source':<35} {'File':<35}")
print("-" * 70)
print(f"{'NBS Electricity Generation':<35} {'china_electricity_generation.csv':<35}")
print(f"{'NBS Energy Source Mix':<35} {'china_energy_mix.csv':<35}")
print(f"{'NEA Solar Capacity':<35} {'china_solar_capacity.csv':<35}")
print(f"{'NEA Wind Capacity':<35} {'china_wind_capacity.csv':<35}")
print("-" * 70)
print(f"\n🕒 Fetched at: {datetime.utcnow().isoformat()} UTC")
print(f"\n✅ All CSV files saved — ready for Spark pipeline (Week 2)!")
print(f"\n🎯 THESIS NOTE:")
print(f"   Source: National Bureau of Statistics (NBS) + National Energy Administration (NEA)")
print(f"   Via: chinadata.live (official government statistics aggregator)")
print(f"   Citation: National Bureau of Statistics of China, 2025")

🔄 Fetching China energy data from NBS...

❌ Electricity Generation failed: All arrays must be of the same length

❌ Energy Mix failed: All arrays must be of the same length

❌ Solar Capacity failed: All arrays must be of the same length

❌ Wind Capacity failed: All arrays must be of the same length

📊 WEEK 1 DATASET SUMMARY
Source                              File                               
----------------------------------------------------------------------
NBS Electricity Generation          china_electricity_generation.csv   
NBS Energy Source Mix               china_energy_mix.csv               
NEA Solar Capacity                  china_solar_capacity.csv           
NEA Wind Capacity                   china_wind_capacity.csv            
----------------------------------------------------------------------

🕒 Fetched at: 2026-09-18T16:06:09.472255 UTC

✅ All CSV files saved — ready for Spark pipeline (Week 2)!

🎯 THESIS NOTE:
   Source: National Bureau of Statistics (NBS) + N

/tmp/ipykernel_1759/1422212896.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"\n🕒 Fetched at: {datetime.utcnow().isoformat()} UTC")
